<a href="https://colab.research.google.com/github/pk-790324/vector_database_for_rag/blob/main/chunking_method_in_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!pip install weaviate-client
from typing import List
import requests
import re
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.util import generate_uuid5
import tqdm
from weaviate.classes.query import Filter

# Downloading the data

In [4]:
url = "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/what-is-git.asc"
source_text = requests.get(url).text

In [6]:
print(source_text[:1000])

[[what_is_git_section]]
=== What is Git?

So, what is Git in a nutshell?
This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.
As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.
Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))

==== Snapshots, Not Differences

The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.
Conceptually, most other systems store information as a list of file-based changes.
These other systems (CVS, Subversion, Perforce, and so o

# Fixed size chunking

In [13]:
def get_chunks_fixed_size(text:str,chunk_size:int)->List[str]:
  text_word=text.split()
  chunks=[]
  for i in range(0,len(text_word),chunk_size):
    chunk_word=text_word[i:i+chunk_size]
    chunk=" ".join(chunk_word)
    chunks.append(chunk)
  return chunks



In [14]:
fixed_size_chunks=get_chunks_fixed_size(source_text,chunk_size=100)

In [15]:
len(fixed_size_chunks)

15

In [17]:
fixed_size_chunks[0:3]

["[[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool. Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in",
 'a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce))) ==== Snapshots, Not Differences The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data. Conceptually, most other systems store information as a list of file-based changes. These other systems (CVS, Subversion, Perforce, and s

# chunking with overlap

In [18]:
def get_chunks_fixed_size_with_overlap(text:str,chunk_size:int,overlap_fraction:float)->List[str]:
  text_word=text.split()
  overlap_int=int(chunk_size*overlap_fraction)
  chunks=[]
  for i in range(0,len(text_word),chunk_size):
    chunk_word=text_word[max(i-overlap_int,0):i+chunk_size]
    chunk=" ".join(chunk_word)
    chunks.append(chunk)
  return chunks

In [24]:
overlap_chunks=get_chunks_fixed_size_with_overlap(source_text,30,0.1)

In [26]:
overlap_chunks[0:3]

['[[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of',
 'the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of the things you may know about',
 "may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool. Even though Git's user interface is fairly similar to"]

# Variable size chunking -Recursive character splitting

### The simplest one is to split into paragraph(/n/n)

In [30]:
def get_chunks_by_paragraph(source_text:str)->List[str]:
  return source_text.split('\n\n')

Another way in this context is to split into sections.As you can see inspecting the text,sections are divided with \n== markers

In [31]:
#split the text by asciidoc section markers
def get_chunks_by_asciidoc_sections(source_text:str)->List[str]:
  return source_text.split('\n==')

In [32]:
for marker in ['\n\n','\n==']:
  chunks=source_text.split(marker)
  print(f'\n Using the marker:{repr(marker)}-{len(chunks)} chunks returned')
  for i in range(3):
    print(f'Chunk {i+1}:{repr(chunks[i])}')


 Using the marker:'\n\n'-31 chunks returned
Chunk 1:'[[what_is_git_section]]\n=== What is Git?'
Chunk 2:"So, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))"
Chunk 3:'==== Snapshots, Not Differences'

 Using the marker:'\n=='-7 chunks returned
Chunk 1:'[[what_is_git_section]]'
Chunk 2:"= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb, because if